# 91 — Campaign B GPU M3 batch (parallel with 89/90)

89 = mass explore（CPU）、90 = lineage advance（CPU）、**91 = staged M3（GPU）**。

- `READY_FOR_M3` を `q_upper` 昇順でキュー
- 共有 M2（package-local `audits/m2_shared_parent.json`）を親にする
- GPU は **1 本ずつ**（resume 可）
- `NOT_CERTIFIED` / production M6 禁止
- 未完了なら同じノートを再実行（同じ child `M3-` run id）


In [ ]:
from pathlib import Path
import os, sys, json

BUNDLE_NAME = 'validated_4d_su2_rg_codex_bundle'
explicit = os.environ.get('VALIDATED_RG_PROJECT_ROOT')
candidates = ([Path(explicit)] if explicit else []) + [
    Path.cwd(), Path.cwd().parent, Path.cwd() / BUNDLE_NAME,
    Path('/notebooks') / BUNDLE_NAME, Path('/notebooks'),
]
PROJECT_ROOT = next((
    p.expanduser().resolve()
    for p in candidates
    if (p.expanduser() / 'src' / 'campaign_b' / 'gpu_m3_batch.py').is_file()
), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Pull latest main; src/campaign_b/gpu_m3_batch.py not found.')
PERSIST_ROOT = Path(
    os.environ.get('VALIDATED_RG_PERSIST_ROOT', '/storage/validated_4d_su2_rg')
).expanduser().resolve()
os.environ['VALIDATED_RG_PROJECT_ROOT'] = str(PROJECT_ROOT)
os.environ['VALIDATED_RG_PERSIST_ROOT'] = str(PERSIST_ROOT)
os.environ.setdefault('VALIDATED_RG_PERSIST_ACK', 'I_CONFIRM_THIS_PATH_IS_PERSISTENT')
os.environ.setdefault('VALIDATED_RG_M3_ALLOW_CODE_DRIFT', '1')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print('PROJECT_ROOT', PROJECT_ROOT)
print('PERSIST_ROOT', PERSIST_ROOT)
print('torch', torch.__version__, 'CUDA', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Notebook 91 requires CUDA GPU.')
print('GPU', torch.cuda.get_device_properties(0).name)


In [ ]:
from src.runtime import validate_persistent_root
from src.campaign_b.gpu_m3_batch import list_gpu_m3_queue, run_gpu_m3_batch

validate_persistent_root()
# Knobs — raise MAX_SESSIONS to keep the GPU busy across resumes:
MAX_SESSIONS = 8          # each session can return at ~6h checkpoint
MAX_QUEUE = 100           # rank by q_upper; process top first
ONLY_CAMPAIGN = None      # e.g. 'M7-20260720T145852Z-b-b1d8f60e0777'

QUEUE = list_gpu_m3_queue(
    PERSIST_ROOT,
    max_candidates=MAX_QUEUE,
    only_campaign_run_id=ONLY_CAMPAIGN,
)
print(json.dumps({
    'queue_size': len(QUEUE),
    'top10': [
        {
            'candidate_id': r.get('candidate_id'),
            'q_upper': r.get('q_upper'),
            'gpu_status': r.get('gpu_status'),
            'package': r.get('package'),
        }
        for r in QUEUE[:10]
    ],
}, indent=2, ensure_ascii=False, default=str))


In [ ]:
SESSION = run_gpu_m3_batch(
    persistent_root=PERSIST_ROOT,
    project_root=PROJECT_ROOT,
    max_sessions=MAX_SESSIONS,
    max_queue=MAX_QUEUE,
    only_campaign_run_id=ONLY_CAMPAIGN,
)
print(json.dumps({
    'session_id': SESSION.get('session_id'),
    'queue_size': SESSION.get('queue_size'),
    'sessions_ok': SESSION.get('sessions_ok'),
    'sessions_error': SESSION.get('sessions_error'),
    'm3_complete': SESSION.get('m3_complete'),
    'm3_checkpoint': SESSION.get('m3_checkpoint'),
    'best_queued_q': SESSION.get('best_queued_q'),
    'errors': SESSION.get('errors'),
    'ledger': str(PERSIST_ROOT / 'campaign_b' / '_gpu_m3' / 'LATEST_GPU_M3_SESSION.json'),
    'certification_status': SESSION.get('certification_status'),
    'claim_scope': SESSION.get('claim_scope'),
    'note': SESSION.get('note'),
}, indent=2, ensure_ascii=False, default=str))
print('Watch GPU: nvidia-smi -l 1')
